# Validated structured tool calls

A complete, deterministic course companion.

## Learning objectives

- Declare a tool with a compact JSON-schema-compatible shape.
- Validate a model-shaped call before dispatch.
- Convert the tool result into a final response.

## Setup

The offline dispatcher is standard Python. It validates the call boundary rather than trusting structured data solely because it resembles JSON.


For the optional live cell, install the dependency with `python -m pip install smolagents`.

## Complete implementation

Run the next cell to build the offline demonstration.

In [1]:
tool_schema = {
    "name": "catering_quote",
    "description": "Quote catering from a guest count and per-person price.",
    "parameters": {
        "type": "object",
        "properties": {
            "guests": {"type": "integer", "minimum": 1},
            "price_per_guest": {"type": "integer", "minimum": 1},
        },
        "required": ["guests", "price_per_guest"],
        "additionalProperties": False,
    },
}

tool_call = {
    "name": "catering_quote",
    "arguments": {"guests": 12, "price_per_guest": 30},
}

def validate_call(call, schema):
    if call.get("name") != schema["name"]:
        raise ValueError("Unknown tool name.")
    arguments = call.get("arguments", {})
    parameters = schema["parameters"]
    if set(arguments) != set(parameters["required"]):
        raise ValueError("Arguments do not match the declared schema.")
    for name, value in arguments.items():
        rule = parameters["properties"][name]
        if rule["type"] == "integer" and (type(value) is not int or value < rule["minimum"]):
            raise ValueError(f"Invalid argument: {name}")
    return arguments

def catering_quote(guests, price_per_guest):
    return {"currency": "EUR", "total": guests * price_per_guest}

tool_registry = {"catering_quote": catering_quote}
validated_arguments = validate_call(tool_call, tool_schema)
tool_result = tool_registry[tool_call["name"]](**validated_arguments)
final_response = f"The catering quote is {tool_result['total']} {tool_result['currency']}."

## Deterministic assertions

These checks make the lesson reproducible without a model or network connection.

In [2]:
assert tool_call["name"] == "catering_quote"
assert validated_arguments == {"guests": 12, "price_per_guest": 30}
assert tool_result == {"currency": "EUR", "total": 360}
assert final_response == "The catering quote is 360 EUR."

print("Offline assertions passed.")

Offline assertions passed.


## Results interpretation

Schema declaration, validation, dispatch, and response formatting are separate stages. This separation makes malformed or unexpected model output fail before it reaches application code.

## Credentials

The offline call uses no credentials. The live `ToolCallingAgent` example checks `HF_TOKEN` and does not reveal it.

In [3]:
import os

live_model_provider = os.getenv("LIVE_MODEL_PROVIDER", "huggingface").strip().casefold()
hf_token = os.getenv("HF_TOKEN")
if live_model_provider != "local" and not hf_token:
    print("SKIPPED: Set HF_TOKEN to run this optional smolagents example.")
else:
    try:
        from smolagents import ToolCallingAgent, tool
        if live_model_provider == "local":
            from smolagents import TransformersModel
        else:
            from smolagents import InferenceClientModel
    except ImportError:
        print("SKIPPED: smolagents is not installed; run `python -m pip install smolagents`.")
    else:
        if live_model_provider == "local":
            model = TransformersModel(
                model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct",
                device_map="cpu",
                torch_dtype="auto",
                max_new_tokens=256,
            )
        else:
            model = InferenceClientModel(
                model_id="Qwen/Qwen2.5-Coder-32B-Instruct", token=hf_token, tool_choice="auto"
            )
        @tool
        def live_catering_quote(guests: int, price_per_guest: int) -> int:
            """Return a quote total.

            Args:
                guests: Positive number of guests.
                price_per_guest: Positive whole-unit price for each guest.
            """
            return guests * price_per_guest

        live_agent = ToolCallingAgent(tools=[live_catering_quote], model=model, max_steps=3)
        live_result = live_agent.run("Quote 12 guests at 30 EUR each.")
        print(live_result)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Quote 12 guests at 30 EUR each.                                                                                 │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen2.5-Coder-1.5B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'live_catering_quote' with arguments: {'guests': 12, 'price_per_guest': 30}                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 360

... (3 outputs left out)

Observations: 360

Final answer: 360

[Step 2: Duration 42.40 seconds| Input tokens: 2,451 | Output tokens: 75]

360


## Completion checklist

- [ ] Declared the tool schema.
- [ ] Validated the structured call.
- [ ] Dispatched the tool and formatted its result.

## Attribution

Original validation example informed by the official [smolagents agent reference](https://huggingface.co/docs/smolagents/reference/agents) and [tool reference](https://huggingface.co/docs/smolagents/reference/tools). No official notebook is copied. Notebook code is released under the Apache-2.0 metadata license.